# Prueba Intertrimestral

**Nombre:**  Alejandra
**Apellidos:**  Botín Lehm

**Tiempo de la prueba:** 2 horas  
**Asignatura:** Desarrollo de Aplicaciones para la Visualización de Datos  
**Fecha:** 19 de octubre de 2027  

## Instrucciones

Escribe **código limpio y autoexplicativo**.  
- Se pueden utilizar **los materiales de clase**.  
- Se puede utilizar **internet** para búsqueda de dudas y documentación.  
- **No se puede utilizar ningún tipo de LLM** (ChatGPT, Copilot, Gemini, etc.).  
- **No se puede utilizar mensajería instantánea**.  
- Al finalizar, **sube tu notebook a GitHub** y envía el enlace del fichero según las instrucciones del profesor.

**[Subé el enlace aquí](https://forms.cloud.microsoft/e/vSwHis0f1B)**

## Dataset “City Micromobility Hourly”

El fichero `city_micromobility_hourly.csv` contiene **alquileres horarios** de bicicletas y patinetes compartidos en una ciudad (ene 2025 – dic 2026): meteorología, tipo de día, usuarios ocasionales/abonados, duración media, ingresos y una etiqueta de alta demanda.

Hay un `.txt` con la descripción de las variables en la misma carpeta.


## Inicialización de librerías


In [37]:
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
import plotly.express as px
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, accuracy_score, confusion_matrix

# Ejercicio 1 — Programación (2 puntos)

a) (0.6 pts) Crea una función `rental_fee(duration_min, bike_type, is_member)` que calcule la tarifa de un viaje en euros, redondeada a **dos decimales**, con estas reglas:

- Tarifa base: **1.00 €** si `bike_type == "classic"`, **1.50 €** si `bike_type == "electric"`.
- Coste por minuto: **0.05 €/min** (classic) o **0.08 €/min** (electric).
- Si `is_member` es `True`, aplica un **20 % de descuento** sobre el total.
- Si `bike_type` no es `"classic"` ni `"electric"`, lanza `ValueError`.

Ejemplo orientativo: `rental_fee(20, "classic", False)` → `2.00`.


In [38]:
def rental_fee(duration_min, bike_type, is_member):
    if bike_type == "classic":
        precio = 1.00 + 0.05 * duration_min
    elif bike_type == "electric":
        precio = 1.50 + 0.08 * duration_min
    else:
        raise ValueError("El tipo debe ser classic o electric")

    if is_member:
        precio = precio * 0.80
    return round(precio, 2)

print(rental_fee(20, "classic", False))
print(rental_fee(20, "electric", True))

2.0
2.48


b) (0.6 pts) Crea una función `demand_label(total_rentals, threshold)` que devuelva `"Alta"` si `total_rentals >= threshold`, `"Media"` si está entre `threshold/2` (incluido) y `threshold` (excluido), y `"Baja"` en caso contrario.


In [39]:
def demand_label(total_rentals, threshold):
    if total_rentals >= threshold:
        return "Alta"
    elif total_rentals >= threshold / 2:
        return "Media"
    else:
        return "Baja"

print(demand_label(40, 40))
print(demand_label(20, 40))
print(demand_label(10, 40))

Alta
Media
Baja


c) (0.6 pts) Genera un DataFrame llamado `tripsDemo` con **al menos 10 filas** y columnas:

- `duration_min` (float)
- `bike_type` (`"classic"` o `"electric"`)
- `is_member` (bool)
- `fee` (calculada con `rental_fee`)
- `total_rentals` (int, inventado)
- `demand` (calculada con `demand_label`, usando por ejemplo `threshold=40`)

Muestra el DataFrame completo.


In [40]:
tripsDemo = pd.DataFrame({
    "duration_min": [5.0, 10.0, 15.0, 20.0, 25.0, 30.0, 35.0, 40.0, 45.0, 50.0],
    "bike_type": ["classic", "electric"] * 5,
    "is_member": [True, False] * 5,
    "total_rentals": [5, 10, 15, 20, 25, 30, 35, 40, 45, 50]
})

tripsDemo["fee"] = [rental_fee(d, b, m) for d, b, m in zip(
    tripsDemo["duration_min"], tripsDemo["bike_type"], tripsDemo["is_member"])]
tripsDemo["demand"] = tripsDemo["total_rentals"].apply(lambda x: demand_label(x, 40))

display(tripsDemo)
print(tripsDemo.dtypes)

,duration_min,bike_type,is_member,total_rentals,fee,demand
0,5.0,classic,True,5,1.0,Baja
1,10.0,electric,False,10,2.3,Baja
2,15.0,classic,True,15,1.4,Baja
3,20.0,electric,False,20,3.1,Media
4,25.0,classic,True,25,1.8,Media
5,30.0,electric,False,30,3.9,Media
6,35.0,classic,True,35,2.2,Media
7,40.0,electric,False,40,4.7,Alta
8,45.0,classic,True,45,2.6,Alta
9,50.0,electric,False,50,5.5,Alta


duration_min     float64
bike_type         object
is_member           bool
total_rentals      int64
fee              float64
demand            object
dtype: object


d) (0.2 pts) Integra ambas funciones en una clase `MicromobilityTools` que, a partir de listas/arrays de `duration_min`, `bike_type` e `is_member`, calcule las tarifas y (opcionalmente) etiquete demanda a partir de una lista de `total_rentals` y un `threshold`.


In [41]:
class MicromobilityTools:
    def __init__(self, duration_min, bike_type, is_member):
        self.datos = pd.DataFrame({
            "duration_min": duration_min,
            "bike_type": bike_type,
            "is_member": is_member
        })

    def tabla(self, total_rentals=None, threshold=40):
        tabla = self.datos.copy()
        tabla["fee"] = [rental_fee(d, b, m) for d, b, m in zip(
            tabla["duration_min"], tabla["bike_type"], tabla["is_member"])]
        if total_rentals is not None:
            tabla["total_rentals"] = total_rentals
            tabla["demand"] = tabla["total_rentals"].apply(
                lambda x: demand_label(x, threshold))
        return tabla

herramientas = MicromobilityTools([10.0, 20.0], ["classic", "electric"], [True, False])
display(herramientas.tabla([15, 50]))

,duration_min,bike_type,is_member,fee,total_rentals,demand
0,10.0,classic,True,1.2,15,Baja
1,20.0,electric,False,3.1,50,Alta


## Ejercicio 2 — Exploración y visualización (3 puntos)

a) (0.75 pts) Carga `city_micromobility_hourly.csv` en un DataFrame `mobilityData`. Muestra las 5 primeras filas y el número de filas y columnas.


In [42]:
ruta = Path("city_micromobility_hourly.csv")
if not ruta.exists():
    ruta = Path("0_Intertrimestral_27") / ruta

mobilityData = pd.read_csv(ruta, parse_dates=["datetime"])
display(mobilityData.head())
print("Filas:", mobilityData.shape[0])
print("Columnas:", mobilityData.shape[1])

,datetime,year,month,day,hour,weekday,season,is_holiday,is_weekend,is_workingday,...,wind_kmh,precip_mm,casual_users,member_users,total_rentals,avg_trip_min,avg_distance_km,electric_share,revenue_eur,high_demand
0,2025-01-01 00:00:00,2025,1,1,0,2,Winter,1,0,0,...,9.7,0.0,2,8,10,23.3,4.94,0.391,10.83,0
1,2025-01-01 01:00:00,2025,1,1,1,2,Winter,1,0,0,...,2.8,0.0,1,5,6,17.3,3.92,0.327,5.35,0
2,2025-01-01 02:00:00,2025,1,1,2,2,Winter,1,0,0,...,9.0,0.0,1,4,5,17.3,4.24,0.405,4.77,0
3,2025-01-01 03:00:00,2025,1,1,3,2,Winter,1,0,0,...,5.4,0.0,2,10,12,17.3,3.98,0.408,10.88,0
4,2025-01-01 04:00:00,2025,1,1,4,2,Winter,1,0,0,...,2.4,0.0,1,4,5,21.6,5.90,0.336,5.16,0


Filas: 17520
Columnas: 23


b) (0.75 pts) Crea una función `describeData(dataFrame)` que, para cada columna, devuelva: tipo de dato, número de nulos y porcentaje de nulos.


In [43]:
def describeData(dataFrame):
    return pd.DataFrame({
        "tipo": dataFrame.dtypes,
        "nulos": dataFrame.isna().sum(),
        "porcentaje_nulos": (dataFrame.isna().mean() * 100).round(2)
    })

c) (0.75 pts) Usa `describeData(mobilityData)` y comenta brevemente (2–3 líneas) qué variables tienen más valores faltantes y qué harías con ellas antes de modelar.


In [44]:
resumen = describeData(mobilityData)
display(resumen)
display(resumen[resumen["nulos"] > 0].sort_values("nulos", ascending=False))

,tipo,nulos,porcentaje_nulos
datetime,datetime64[ns],0,0.00
year,int64,0,0.00
month,int64,0,0.00
day,int64,0,0.00
hour,int64,0,0.00
weekday,int64,0,0.00
season,object,0,0.00
is_holiday,int64,0,0.00
is_weekend,int64,0,0.00
is_workingday,int64,0,0.00


,tipo,nulos,porcentaje_nulos
avg_trip_min,float64,255,1.46
humidity,float64,224,1.28
electric_share,float64,182,1.04
wind_kmh,float64,142,0.81


Los nulos se concentran en avg_trip_min (255, 1,46 %), humidity (224, 1,28 %), electric_share (182, 1,04 %) y wind_kmh (142, 0,81 %). Para simplificar, elimino solo las filas con nulos en los predictores elegidos

d) (0.75 pts) Elabora **dos** visualizaciones:

1. Un gráfico de correlación entre variables numéricas relevantes (por ejemplo `temp_c`, `humidity`, `wind_kmh`, `precip_mm`, `casual_users`, `member_users`, `total_rentals`, `revenue_eur`).
2. Un gráfico que muestre la demanda media (`total_rentals`) por `hour` (y, si quieres, separando fin de semana / laborable).

Comenta 2–3 hallazgos (correlaciones, horas punta, efecto del clima, etc.).


In [45]:
variables_numericas = ["temp_c", "humidity", "wind_kmh", "precip_mm",
                       "casual_users", "member_users", "total_rentals", "revenue_eur"]
correlaciones = mobilityData[variables_numericas].corr()
px.imshow(correlaciones, text_auto=".2f", zmin=-1, zmax=1,
          color_continuous_scale="RdBu_r", title="Correlaciones entre variables").show()

por_hora = mobilityData.groupby(["hour", "is_weekend"], as_index=False)["total_rentals"].mean()
por_hora["tipo_dia"] = por_hora["is_weekend"].map({0: "Entre semana", 1: "Fin de semana"})
px.line(por_hora, x="hour", y="total_rentals", color="tipo_dia", markers=True,
        title="Demanda media por hora y tipo de día",
        labels={"hour": "Hora", "total_rentals": "Alquileres medios", "tipo_dia": "Tipo de día"}).show()

print("Correlaciones con los alquileres:")
print(correlaciones["total_rentals"].round(3))
print("Horas con mayor demanda media:")
display(mobilityData.groupby("hour")["total_rentals"].mean().nlargest(4))

Correlaciones con los alquileres:
temp_c           0.441
humidity        -0.169
wind_kmh        -0.048
precip_mm       -0.181
casual_users     0.823
member_users     0.936
total_rentals    1.000
revenue_eur      0.915
Name: total_rentals, dtype: float64
Horas con mayor demanda media:


hour
18    63.824658
17    63.276712
9     62.850685
7     62.815068
Name: total_rentals, dtype: float64

Como conclusión de este apartado, las horas con más alquileres medios son 18, 17 y 9 (que tiene sentido porque al ser entre semana es la hora de entrada y de salida del trabajo). La correlación con temperatura es 0.44, con humedad -0.17 y con precipitación -0.18. Si tratamos el caso de los usuarios abonados son los que mas relacionados están con el total porque forman parte de él, los ingresos también se generan a partir de los viajes. No utilizaría estas variables que están directamente relacionadas con la variable objetivo para anticipar demanda

## Ejercicio 3 — Regresión (2 puntos)

Queremos predecir `total_rentals`.

a) (0.25 pts) Define `targetVar = "total_rentals"` y selecciona **3–5** variables predictoras numéricas razonables (p. ej. `hour`, `temp_c`, `humidity`, `wind_kmh`, `precip_mm`, `is_workingday`, `is_weekend`). Construye `X` e `y`, tratando nulos de forma simple (p. ej. eliminar filas o imputar la mediana).


In [46]:
targetVar = "total_rentals"
variables = ["hour", "temp_c", "humidity", "precip_mm", "is_workingday"]

# Solo elimino filas con nulos en las columnas que voy a usar.
datos_regresion = mobilityData.dropna(subset=variables + [targetVar])
X = datos_regresion[variables] 
y = datos_regresion[targetVar]
print("Filas usadas:", len(X))
print("Filas eliminadas:", len(mobilityData) - len(X))

Filas usadas: 17296
Filas eliminadas: 224


He elegido las variables hora, temperatura, humedad, precipitación y día laborable, porque al mirar el diagrama de correlacion de las variables y el grafico de la demanda por dia y hora, estas son las variables que mejor van a explicar la variable objetivo que es total_rentals

b) (0.25 pts) Divide en entrenamiento (70 %) y prueba (30 %) con `random_state=42`.


In [47]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42)
print("Train:", len(X_train))
print("Test:", len(X_test))

Train: 12107
Test: 5189


c) (0.75 pts) Entrena un `LinearRegression`, calcula **R²** y **MSE** en el conjunto de prueba y comenta el resultado en 1–2 líneas.


In [48]:
lineal = LinearRegression()
# Filtro local de los avisos de matmul observados en este entorno.
# Los demás warnings siguen visibles; compruebo que el resultado sea válido.
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=RuntimeWarning,
                            message=r"(divide by zero|overflow|invalid value) encountered in matmul",
                            module=r"sklearn\..*")
    lineal.fit(X_train, y_train)
    pred_lineal = lineal.predict(X_test)

assert np.isfinite(lineal.coef_).all() and np.isfinite(pred_lineal).all(), "Resultado no finito"

r2_lineal = r2_score(y_test, pred_lineal)
mse_lineal = mean_squared_error(y_test, pred_lineal)
print("R2:", round(r2_lineal, 3))
print("MSE:", round(mse_lineal, 3))

R2: 0.32
MSE: 403.658


La regresión lineal obtiene R2 = 0.32 y MSE = 403.658. Su R2 es bastante bajo y el MSE parece también alto. Un efecto lineal de la hora no puede representar varios picos de demanda diarios. Por lo que vamos a usar otro modelo Random Forest en el siguiente apartado para ver si puede explicar mejor este conjunto de datos porque este modelo de regresión lineal no ha sido el mejor

d) (0.75 pts) Entrena un `RandomForestRegressor` (mismo split) y compara R² y MSE con la regresión lineal. ¿Cuál funciona mejor y por qué crees que es así?


In [49]:
bosque = RandomForestRegressor(random_state=42)
bosque.fit(X_train, y_train)
pred_bosque = bosque.predict(X_test)

r2_bosque = r2_score(y_test, pred_bosque)
mse_bosque = mean_squared_error(y_test, pred_bosque)
resultados = pd.DataFrame({
    "modelo": ["Regresión lineal", "Random Forest"],
    "R2": [r2_lineal, r2_bosque],
    "MSE": [mse_lineal, mse_bosque]
})
display(resultados.round(3))
print("Mejor modelo:", resultados.loc[resultados["MSE"].idxmin(), "modelo"])

,modelo,R2,MSE
0,Regresión lineal,0.320,403.658
1,Random Forest,0.882,69.742


Mejor modelo: Random Forest


Como se suponía, al comparar regresión lineal con random forest, el mejor de los dos en este test es Random Forest. Random Forest obtiene R2 = 0.882 y MSE = 69.742. Random Forest puede recoger varios picos horarios y combinaciones no lineales entre hora, clima y día laborable. Estos dos modelos se han comparado usando exactamente las mismas filas de entrenamiento y prueba

## Ejercicio 4 — Clasificación (2 puntos)

Usa la columna `high_demand` (1 = demanda alta, 0 = no).

a) (0.25 pts) Comprueba el balance de clases (`value_counts` o proporción) de `high_demand`.


In [50]:
print("Número de observaciones por clase:")
print(mobilityData["high_demand"].value_counts().sort_index())
print("Porcentaje por clase:")
print((mobilityData["high_demand"].value_counts(normalize=True).sort_index() * 100).round(2))

Número de observaciones por clase:
high_demand
0    12071
1     5449
Name: count, dtype: int64
Porcentaje por clase:
high_demand
0    68.9
1    31.1
Name: proportion, dtype: float64


La demanda alta representa el 31.1 % de las horas y el resto es demanda normal

b) (0.75 pts) Entrena un `LogisticRegression` para predecir `high_demand` usando variables de contexto (hora, clima, tipo de día, etc.). Recuerda preparar `X`/`y` y el split train/test.


In [51]:
datos_clasificacion = mobilityData.dropna(subset=variables + ["high_demand"]) # Reutilizo las cinco variables de la regresión lineal (ejercicio 3)
X_clas = datos_clasificacion[variables]
y_clas = datos_clasificacion["high_demand"]

X_train_clas, X_test_clas, y_train_clas, y_test_clas = train_test_split(
    X_clas, y_clas, test_size=0.30, random_state=42, stratify=y_clas)

escalador = StandardScaler()
X_train_escalado = escalador.fit_transform(X_train_clas)
X_test_escalado = escalador.transform(X_test_clas)

logistica = LogisticRegression(max_iter=1000)
# Mismo filtro específico, limitado al ajuste y predicción de este modelo.
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=RuntimeWarning,
                            message=r"(divide by zero|overflow|invalid value) encountered in matmul",
                            module=r"sklearn\..*")
    logistica.fit(X_train_escalado, y_train_clas)
    pred_clas = logistica.predict(X_test_escalado)

assert np.isfinite(logistica.coef_).all() and np.isfinite(pred_clas).all(), "Resultado no finito"

c) (0.25 pts) Calcula la **accuracy** y muestra la **matriz de confusión**.


In [52]:
accuracy = accuracy_score(y_test_clas, pred_clas)
print("Accuracy:", round(accuracy, 3))
clase_mayoritaria = y_train_clas.mode()[0]
referencia = (y_test_clas == clase_mayoritaria).mean()
print("Accuracy si siempre predigo la clase mayoritaria:", round(referencia, 3))

matriz = confusion_matrix(y_test_clas, pred_clas, labels=[0, 1])
display(pd.DataFrame(matriz, index=["Real normal", "Real alta"],
                     columns=["Predicción normal", "Predicción alta"]))
px.imshow(matriz, text_auto=True, x=["Normal", "Alta"], y=["Normal", "Alta"],
          labels={"x": "Predicción", "y": "Real"},
          color_continuous_scale="Blues", title="Matriz de confusión").show()

Accuracy: 0.757
Accuracy si siempre predigo la clase mayoritaria: 0.689


,Predicción normal,Predicción alta
Real normal,3255,320
Real alta,939,675


d) (0.75 pts) Interpreta qué factores parecen asociarse más a la alta demanda (coeficientes del modelo logístico y/o importancia si usas un árbol; o argumentación basada en tus gráficos del ejercicio 2).


In [53]:
coeficientes = pd.Series(logistica.coef_[0], index=variables)
orden = coeficientes.abs().sort_values(ascending=False).index
coeficientes = coeficientes.loc[orden]
display(coeficientes.to_frame("coeficiente_estandarizado").round(3))
px.bar(x=coeficientes.index, y=coeficientes.values, title="Factores asociados a alta demanda",
       labels={"x": "Variable", "y": "Coeficiente estandarizado"}).show()
print("Mayor coeficiente en valor absoluto:", coeficientes.index[0])

,coeficiente_estandarizado
precip_mm,-0.808
temp_c,0.763
hour,0.340
humidity,-0.243
is_workingday,-0.241


Mayor coeficiente en valor absoluto: precip_mm


Como se observa en este gráfico, y como se puede comparar coeficientes porque están normalizados. Se puede ver que los coeficientes más significativos asociados con la alta demanda son las precipitaciones, que si llueve es muy improbable que la gente alquile una bicicleta/patinete, la temperatura, que si hace buen tiempo y se está agradable, es bastante posible que la gente alquile una bicicleta/patinete y, en menor medida, la hora, porque en las horas punta de entrada y salida del trabajo se alquilan mas bicicletas/patinetes tambien

## Ejercicio 5 — Conclusión (1 punto)

En un máximo de **300 palabras**, resume:

* Qué factores (hora, clima, tipo de día, etc.) parecen mover más la demanda de micromovilidad.  
* Qué modelo predictivo te ha funcionado mejor y por qué.  
* Qué mejorarías con más tiempo o datos (más ciudades, eventos, precio dinámico, etc.).


Al finalizar:

**[Subé el enlace aquí](https://forms.cloud.microsoft/e/vSwHis0f1B)**



In [54]:
# para respaldar la conclusión con números
print(resultados.round(3).to_string(index=False))
print("Accuracy logística:", round(accuracy, 3))

          modelo    R2     MSE
Regresión lineal 0.320 403.658
   Random Forest 0.882  69.742
Accuracy logística: 0.757


La demanda varía con la hora y con el tipo de día (laborable o no), como lo muestran las curvas de demanda horarias. La temperatura tiene una correlación de 0.44 con los alquileres y la precipitación de -0.18. Son asociaciones de estos datos, no efectos causales. El coeficiente normalizado con mayor valor absoluto en la logística es precip_mm.

El modelo de regresión que mejor se ajusta a los datos es Random Forest, obteniendo un R2 de 0.882 y un MSE de 69.74. Random Forest permite representar picos de demanda e interacciones que una regresión lineal no recoge. La logística logra una precisión del 75.7%, frente al 68.9% de la referencia.

Como recomendación final personal, diría que hay que preparar más vehículos en las horas punta identificadas (a la entrada y a la salida del trabajo) y ajustar el servicio a las previsiones meteorológicas (si se ve que va a llover mucho se puede esperar que nadie alquile una bicicleta/patinete para tener el riesgo de caerse). Con más tiempo, reservaría las últimas fechas para evaluar predicciones futuras, probaría codificar cada hora como categoría (especialmente las horas punta) y añadiría nuevas variables como eventos, ubicaciones y disponibilidad de vehículos. También revisaría los errores de demanda alta, compararía más ciudades y comprobaría el comportamiento con datos reales. La partición aleatoria del ejercicio puede ofrecer resultados más favorables que una evaluación temporal.